# Version17 train — Member 3 only
Reads shared Data_V3_all_columns without changing any source file. Removes attempted rows in memory before validation.
Behavioural features are calculated within the existing splits; original random split membership is preserved.


In [5]:
from pathlib import Path
import sys, json, random
import numpy as np
import pandas as pd
import joblib
import tensorflow as tf
import matplotlib.pyplot as plt
from sklearn.metrics import (confusion_matrix, ConfusionMatrixDisplay, accuracy_score,
                             precision_score, recall_score, f1_score, roc_auc_score,
                             average_precision_score, classification_report)
ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks': ROOT = ROOT.parent
if not (ROOT/'data'/'processed').exists():
    raise FileNotFoundError('Run from the project root or notebooks folder')
sys.path.insert(0,str(ROOT/'notebooks'))
from version17_common import (
    VERSION, UNKNOWN, add_behavioural_features, BEHAVIOURAL_FEATURES, validate_frame, fit_preprocessing, transform,
    ports, reconstruction_mse, calibrate, local_thresholds, verify_behavioural_inputs, load_clean_split)
DATA_DIR=ROOT/'data'/'processed'/'Data_V3_full'
MODEL_DIR=ROOT/'saved-models'/'version17'
RESULTS_DIR=ROOT/'outputs'/'results'/'version17_unknown_test'
MODEL_DIR.mkdir(parents=True,exist_ok=True)
RESULTS_DIR.mkdir(parents=True,exist_ok=True)
SEED=42
random.seed(SEED); np.random.seed(SEED); tf.keras.utils.set_random_seed(SEED)

# Read existing team data without overwriting CSV/Parquet. Attempted rows are removed in memory.
load_csv = lambda name: load_clean_split(DATA_DIR, name, benign_only=name.startswith('autoencoder_benign_'))


In [6]:
train=load_csv('autoencoder_benign_train')
val=load_csv('autoencoder_benign_validation')
train=validate_frame(train,'train',benign_only=True)
val=validate_frame(val,'validation',benign_only=True)
train=add_behavioural_features(train)
val=add_behavioural_features(val)
prep=fit_preprocessing(train)
x_train=transform(train,prep)
x_val=transform(val,prep)
print('Model feature count:',prep['input_dim'])
print('Input columns:',prep['columns'])
print('Train/validation:',x_train.shape,x_val.shape)

verify_behavioural_inputs(train, val, prep, x_train, x_val)
print('Excluded fields:', prep['excluded_fields'])
print('Numeric model features:', len(prep['columns']))
print('Non-numeric columns preserved as metadata:', prep['non_numeric_columns'])
print('Training-constant/empty columns:', prep['constant_or_empty_columns'])

assert all(f in prep['columns'] for f in BEHAVIOURAL_FEATURES), 'Behavioural inputs missing'


autoencoder_benign_train: read 999,418 from autoencoder_benign_train.csv; removed 3,164 attempted; retained 996,254
autoencoder_benign_validation: read 214,161 from autoencoder_benign_validation.parquet; removed 635 attempted; retained 213,526
Model feature count: 86
Input columns: ['Src Port', 'Dst Port', 'Protocol', 'Flow Duration', 'Total Fwd Packet', 'Total Bwd packets', 'Total Length of Fwd Packet', 'Total Length of Bwd Packet', 'Fwd Packet Length Max', 'Fwd Packet Length Min', 'Fwd Packet Length Mean', 'Fwd Packet Length Std', 'Bwd Packet Length Max', 'Bwd Packet Length Min', 'Bwd Packet Length Mean', 'Bwd Packet Length Std', 'Flow Bytes/s', 'Flow Packets/s', 'Flow IAT Mean', 'Flow IAT Std', 'Flow IAT Max', 'Flow IAT Min', 'Fwd IAT Total', 'Fwd IAT Mean', 'Fwd IAT Std', 'Fwd IAT Max', 'Fwd IAT Min', 'Bwd IAT Total', 'Bwd IAT Mean', 'Bwd IAT Std', 'Bwd IAT Max', 'Bwd IAT Min', 'Fwd PSH Flags', 'Bwd PSH Flags', 'Fwd RST Flags', 'Bwd RST Flags', 'Fwd Header Length', 'Bwd Header Leng

In [7]:
from tensorflow.keras import Model, Input
from tensorflow.keras.layers import Dense
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

def make_autoencoder(dim):
    inp=Input(shape=(dim,))
    x=Dense(48,activation='relu')(inp)
    x=Dense(24,activation='relu')(x)
    x=Dense(12,activation='linear',name='bottleneck')(x)
    x=Dense(24,activation='relu')(x)
    x=Dense(48,activation='relu')(x)
    out=Dense(dim,activation='linear')(x)
    model=Model(inp,out)
    model.compile(optimizer=tf.keras.optimizers.Adam(0.001),loss='mse')
    return model

seeds=[42,43,44]
models=[]; histories=[]
for seed in seeds:
    tf.keras.backend.clear_session()
    tf.keras.utils.set_random_seed(seed)
    model=make_autoencoder(x_train.shape[1])
    hist=model.fit(x_train,x_train,validation_data=(x_val,x_val),
                   epochs=100,batch_size=1024,verbose=1,
                   callbacks=[EarlyStopping(monitor='val_loss',patience=10,restore_best_weights=True),
                              ReduceLROnPlateau(monitor='val_loss',patience=4,factor=.5,min_lr=1e-6)])
    models.append(model); histories.append(hist.history)
    print('Seed',seed,'best validation loss',min(hist.history['val_loss']))


Epoch 1/100
973/973 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - loss: 0.2142 - val_loss: 0.0558 - learning_rate: 0.0010
Epoch 2/100
973/973 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - loss: 0.0320 - val_loss: 0.0317 - learning_rate: 0.0010
Epoch 3/100
973/973 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - loss: 0.0244 - val_loss: 0.0269 - learning_rate: 0.0010
Epoch 4/100
973/973 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - loss: 0.0212 - val_loss: 0.0241 - learning_rate: 0.0010
Epoch 5/100
973/973 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - loss: 0.0192 - val_loss: 0.0220 - learning_rate: 0.0010
Epoch 6/100
973/973 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - loss: 0.0179 - val_loss: 0.0204 - learning_rate: 0.0010
Epoch 7/100
973/973 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - loss: 0.0169 - val_loss: 0.0189 - learning_rate: 0.0010
Epoch 8/100
973/973 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - loss: 0.0162 - val_loss: 0.0176 - learning_rate: 0.0010
Epoch 9/100
973/973 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - loss: 0.0156 - val_loss: 0.0168 - learning_rate: 0.0010
E

In [8]:
val_scores=reconstruction_mse(models,x_val)
calibration=calibrate(val_scores,ports(val),target_fpr=.03,min_port=500,min_group=1000)
print('Global threshold:',calibration['global_threshold'])
print('Eligible ports:',len(calibration['port_thresholds']))
print('Eligible groups:',len(calibration['group_thresholds']))
print('Validation global FPR:',np.mean(val_scores>calibration['global_threshold']))
print('Validation local FPR:',np.mean(val_scores>local_thresholds(ports(val),calibration)))


Global threshold: 0.050564720295369625
Eligible ports: 7
Eligible groups: 3
Validation global FPR: 0.030001030319492708
Validation local FPR: 0.034422037597294945


In [9]:
files=[]
for seed,model in zip(seeds,models):
    name=f'autoencoder_{VERSION}_seed{seed}.keras'
    model.save(MODEL_DIR/name); files.append(name)
joblib.dump(prep,MODEL_DIR/'preprocessing.joblib')
(MODEL_DIR/'calibration.json').write_text(json.dumps(calibration,indent=2))
(MODEL_DIR/'selection.json').write_text(json.dumps({
    'version':VERSION,'model_files':files,'input_dim':prep['input_dim'],
    'features':prep['columns'],'calibration_file':'calibration.json'},indent=2))
print('Artifacts saved to',MODEL_DIR)


Artifacts saved to C:\Users\saaral\Desktop\gc\pattern recog\project\]code\Hybrid-Network-Intrusion-Detection\saved-models\version17
